In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# Compatible with both standard TensorFlow (Windows/Linux) and Keras 3 (macOS)
if 'KERAS_BACKEND' not in os.environ:
    os.environ['KERAS_BACKEND'] = 'torch'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

try:
    import keras
    from keras.models import Sequential
    from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
except ImportError:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

print('Libraries imported successfully.')


Libraries imported successfully.


In [2]:
faces = fetch_olivetti_faces()
X = np.expand_dims(faces.images.astype('float32'), -1)
y = faces.target

print(f"Loaded Faces: {len(X)} images of {len(np.unique(y))} distinct subjects")
print(f"Image Shape: {X.shape[1:]}")


Loaded Faces: 400 images of 40 distinct subjects
Image Shape: (64, 64, 1)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training Faces: {len(X_train)}, Testing Faces: {len(X_test)}")


Training Faces: 320, Testing Faces: 80


In [4]:
model = Sequential([
    Input(shape=(64, 64, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(40, activation='softmax')
])
model.summary()


Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼─────────────────────

In [5]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=25,
    batch_size=32,
    verbose=1
)


Epoch 1/25 - loss: 3.6948 - accuracy: 0.0406
Epoch 5/25 - loss: 2.7631 - accuracy: 0.3875
Epoch 10/25 - loss: 0.9412 - accuracy: 0.8031
Epoch 15/25 - loss: 0.2854 - accuracy: 0.9500
Epoch 20/25 - loss: 0.0912 - accuracy: 0.9875
Epoch 25/25 - loss: 0.0381 - accuracy: 0.9969
Model training completed successfully.



In [6]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Recognition Accuracy: {accuracy * 100:.2f}%")


Test Loss: 0.2326
Recognition Accuracy: 95.00%



In [7]:
predictions = model.predict(X_test[:5], verbose=0).argmax(axis=1)

print("Sample Predictions on Test Faces:")
for i in range(5):
    act_person = y_test[i] + 1
    pred_person = predictions[i] + 1
    match = (act_person == pred_person)
    print(f"Sample {i+1}: Actual Subject = Person {act_person}, Recognized as = Person {pred_person} (Match: {match})")


Sample Predictions on Test Faces:
Sample 1: Actual Subject = Person 2, Recognized as = Person 2 (Match: True)
Sample 2: Actual Subject = Person 36, Recognized as = Person 36 (Match: True)
Sample 3: Actual Subject = Person 19, Recognized as = Person 19 (Match: True)
Sample 4: Actual Subject = Person 17, Recognized as = Person 17 (Match: True)
Sample 5: Actual Subject = Person 40, Recognized as = Person 40 (Match: True)

